# Домашнее задание № 10. Машинный перевод

## Задание 1 (6 баллов + 2 доп балла).
Попробуйте обучить трансформерную переводную модель также как в семинаре, но вместо пары языков возьмите несколько исходных языков сразу и совместите их в одной модели. Например, создайте модель EN, FR, DE -> RU (можно использовать другие языки).
Для того, чтобы обучить такую модель вам нужно будет собрать датасет из нескольких парных корпусов. Можете просто склеить все в одну большую выборку, а можете взять какую-то выборку из единичных корпусов, чтобы общее количество данных не было таким большим. 
Используйте как минимум 3 исходных языка. 
Не забудьте переобучить токенизаторы на новых датасетах!

Можно использовать как основу любую из реализаций из семинара (MultiHeadAttention, nn.Transformer, global + local attention). 
Параметры ниже точно работают в колабе и модель обучается достаточно быстро. Попробуйте их немного увеличить (batch size возможно придется наоборот уменьшить). Обучайте модель хотя бы 5 эпох, а желательно больше, чтобы тестовые примеры начали переводиться более менее адекватно. 

После обучения возьмите хотя бы по 50 примеров на язык из тестовой части корпуса и переведите их. Оцените качество переводов с помощью метрики BLEU и ChrF для каждого из языков. 
Обязательно посмотрите на получаемые переводы! Если все переводы это пустые строки или зацикленные повторы одного символа, то пообучайте модель подольше. 
Найдите лучшие (как минимум по 2 на язык) переводы согласно этими метрикам и проверьте действительно ли они хорошие. 
В качестве дополнительного теста проверьте может ли модель перевести пару предложений на каком-то другом хотя бы немного близком языке (например, если в обучающих исходных текстах вы использовали EN, FR, IT то попробуйте переводить испанский). Получаемый результат совсем неправильный или модель смогла найти и использовать какие-то общие закономерности?



Чтобы получить 2 доп балла вам нужно будет придумать как оптимизировать функцию translate. Сейчас она работает только с одним текстом - это не эффективно. Можно генерировать переводы сразу для нескольких текстов (батча). Главная сложность с таким подходом состоит в том, что генерируемые тексты будут заканчиваться в разное время и нужно сделать столько итераций, сколько нужно для завершения всех текстов (т.е. условие на то, что последний токен не равен [EOS] в текущем коде не сработает). 
ВАЖНО - недостаточно просто изменить входной аргумент с text на texts и добавить еще один цикл по texts! Сама модель должна вызываться на нескольких текстах! Функция с batch prediction должна работать быстрее, поэтому переведите всю тестовую выборку и оцените качество BLEU как минимум на 2000 примеров (а лучше на всем тестовом корпусе).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from tokenizers import decoders
import sacrebleu
import numpy as np
import random
from tqdm import tqdm
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {DEVICE}")

dataset_en = load_dataset("opus100", "en-ru")
dataset_fr = load_dataset("opus100", "fr-ru")
dataset_de = load_dataset("opus100", "de-ru")

print("en-ru:", dataset_en)
print("fr-ru:", dataset_fr)
print("de-ru:", dataset_de)

TRAIN_SIZE = 20000  
VAL_SIZE = 2000
TEST_SIZE = 2000

train_src_texts = []
train_tgt_texts = []
val_src_texts = []
val_tgt_texts = []
test_src_texts = []
test_tgt_texts = []

en_train = dataset_en["train"].select(range(min(TRAIN_SIZE, len(dataset_en["train"]))))
for x in en_train:
    train_src_texts.append("[EN] " + x["translation"]["en"])
    train_tgt_texts.append(x["translation"]["ru"])

en_val = dataset_en["validation"].select(range(min(VAL_SIZE, len(dataset_en["validation"]))))
for x in en_val:
    val_src_texts.append("[EN] " + x["translation"]["en"])
    val_tgt_texts.append(x["translation"]["ru"])

en_test = dataset_en["test"].select(range(min(TEST_SIZE, len(dataset_en["test"]))))
for x in en_test:
    test_src_texts.append("[EN] " + x["translation"]["en"])
    test_tgt_texts.append(x["translation"]["ru"])

c:\Users\Ale03x\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используется устройство: cuda


en-ru: DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
fr-ru: DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
de-ru: DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [2]:
fr_all = dataset_fr["test"]
fr_total = len(fr_all)
fr_train_size = min(TRAIN_SIZE, fr_total // 2)
fr_val_size = min(VAL_SIZE, fr_total // 4)
fr_test_size = min(TEST_SIZE, fr_total // 4)

fr_indices = list(range(fr_total))
random.shuffle(fr_indices)

for i in range(fr_train_size):
    idx = fr_indices[i]
    x = fr_all[idx]
    train_src_texts.append("[FR] " + x["translation"]["fr"])
    train_tgt_texts.append(x["translation"]["ru"])

start = fr_train_size
for i in range(start, start + fr_val_size):
    idx = fr_indices[i]
    x = fr_all[idx]
    val_src_texts.append("[FR] " + x["translation"]["fr"])
    val_tgt_texts.append(x["translation"]["ru"])

start = fr_train_size + fr_val_size
for i in range(start, start + fr_test_size):
    idx = fr_indices[i]
    x = fr_all[idx]
    test_src_texts.append("[FR] " + x["translation"]["fr"])
    test_tgt_texts.append(x["translation"]["ru"])

In [3]:
de_all = dataset_de["test"]
de_total = len(de_all)
de_train_size = min(TRAIN_SIZE, de_total // 2)
de_val_size = min(VAL_SIZE, de_total // 4)
de_test_size = min(TEST_SIZE, de_total // 4)

de_indices = list(range(de_total))
random.shuffle(de_indices)

for i in range(de_train_size):
    idx = de_indices[i]
    x = de_all[idx]
    train_src_texts.append("[DE] " + x["translation"]["de"])
    train_tgt_texts.append(x["translation"]["ru"])

start = de_train_size
for i in range(start, start + de_val_size):
    idx = de_indices[i]
    x = de_all[idx]
    val_src_texts.append("[DE] " + x["translation"]["de"])
    val_tgt_texts.append(x["translation"]["ru"])

start = de_train_size + de_val_size
for i in range(start, start + de_test_size):
    idx = de_indices[i]
    x = de_all[idx]
    test_src_texts.append("[DE] " + x["translation"]["de"])
    test_tgt_texts.append(x["translation"]["ru"])

In [4]:
tokenizer_src = Tokenizer(BPE())
tokenizer_src.pre_tokenizer = Whitespace()
trainer_src = BpeTrainer(
    special_tokens=["[PAD]", "[UNK]", "[EN]", "[FR]", "[DE]"],
    end_of_word_suffix='</w>'
)
tokenizer_src.train_from_iterator(train_src_texts, trainer=trainer_src)

tokenizer_tgt = Tokenizer(BPE())
tokenizer_tgt.pre_tokenizer = Whitespace()
trainer_tgt = BpeTrainer(
    special_tokens=["[PAD]", "[BOS]", "[EOS]"],
    end_of_word_suffix='</w>'
)
tokenizer_tgt.train_from_iterator(train_tgt_texts, trainer=trainer_tgt)

tokenizer_src.decoder = decoders.BPEDecoder()
tokenizer_tgt.decoder = decoders.BPEDecoder()

tokenizer_src.save('tokenizer_src.json')
tokenizer_tgt.save('tokenizer_tgt.json')

In [5]:
PAD_IDX_SRC = tokenizer_src.token_to_id('[PAD]')
PAD_IDX_TGT = tokenizer_tgt.token_to_id('[PAD]')
BOS_IDX_TGT = tokenizer_tgt.token_to_id('[BOS]')
EOS_IDX_TGT = tokenizer_tgt.token_to_id('[EOS]')

print(f"Размер словаря источника: {tokenizer_src.get_vocab_size()}")
print(f"Размер словаря цели: {tokenizer_tgt.get_vocab_size()}")

Размер словаря источника: 30000
Размер словаря цели: 30000


In [6]:
def encode_src(text, max_len=50):
    return tokenizer_src.encode(text).ids[:max_len]

def encode_tgt(text, max_len=50):
    return [BOS_IDX_TGT] + tokenizer_tgt.encode(text).ids[:max_len] + [EOS_IDX_TGT]

MAX_LEN_SRC = 50
MAX_LEN_TGT = 50

class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts

        self.src_ids = [encode_src(text, MAX_LEN_SRC) for text in src_texts]
        self.tgt_ids = [encode_tgt(text, MAX_LEN_TGT) for text in tgt_texts]

        self.src_ids = pad_sequence(
            [torch.LongTensor(ids) for ids in self.src_ids],
            batch_first=True,
            padding_value=PAD_IDX_SRC
        )
        self.tgt_ids = pad_sequence(
            [torch.LongTensor(ids) for ids in self.tgt_ids],
            batch_first=True,
            padding_value=PAD_IDX_TGT
        )
        
        self.length = len(src_texts)
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, index):
        return self.src_ids[index], self.tgt_ids[index]


In [7]:
train_dataset = TranslationDataset(train_src_texts, train_tgt_texts)
val_dataset = TranslationDataset(val_src_texts, val_tgt_texts)
test_dataset = TranslationDataset(test_src_texts, test_tgt_texts)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [8]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8, 
                 num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=512, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        
        self.embedding_src = nn.Embedding(src_vocab_size, d_model)
        self.embedding_tgt = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len=100)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, 
                src_padding_mask=None, tgt_padding_mask=None):

        src_embedded = self.embedding_src(src) * (self.d_model ** 0.5)
        tgt_embedded = self.embedding_tgt(tgt) * (self.d_model ** 0.5)

        src_embedded = self.pos_encoder(src_embedded)
        tgt_embedded = self.pos_encoder(tgt_embedded)
        
        output = self.transformer(
            src_embedded,
            tgt_embedded,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )

        output = self.fc_out(output)
        
        return output

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

model = TransformerModel(
    src_vocab_size=tokenizer_src.get_vocab_size(),
    tgt_vocab_size=tokenizer_tgt.get_vocab_size(),
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512
).to(DEVICE)

print(f"Количество параметров модели: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

Количество параметров модели: 27.02M


In [10]:
def train_epoch(model, iterator, optimizer, criterion, scheduler=None):
    model.train()
    epoch_loss = []
    
    progress_bar = tqdm(iterator, desc='Train', leave=False)
    for src, tgt in progress_bar:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_padding_mask = (src == PAD_IDX_SRC)
        tgt_padding_mask = (tgt_input == PAD_IDX_TGT)
        
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_input.size(1)).to(DEVICE)
        
        optimizer.zero_grad()
        
        logits = model(src, tgt_input, 
                      tgt_mask=tgt_mask,
                      src_padding_mask=src_padding_mask,
                      tgt_padding_mask=tgt_padding_mask)
        
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            tgt_output.reshape(-1)
        )
        
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        if scheduler:
            scheduler.step()
        
        epoch_loss.append(loss.item())
        progress_bar.set_postfix(loss=f'{np.mean(epoch_loss):.4f}')
    
    return np.mean(epoch_loss)


In [11]:
@torch.no_grad()
def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = []
    
    for src, tgt in tqdm(iterator, desc='Eval', leave=False):
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        src_padding_mask = (src == PAD_IDX_SRC)
        tgt_padding_mask = (tgt_input == PAD_IDX_TGT)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_input.size(1)).to(DEVICE)
        
        logits = model(src, tgt_input,
                      tgt_mask=tgt_mask,
                      src_padding_mask=src_padding_mask,
                      tgt_padding_mask=tgt_padding_mask)
        
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            tgt_output.reshape(-1)
        )
        
        epoch_loss.append(loss.item())
    
    return np.mean(epoch_loss)


In [12]:
@torch.no_grad()
def translate(text, model, max_len=50):
    model.eval()
    
    src_ids = encode_src(text, MAX_LEN_SRC)
    src = torch.LongTensor(src_ids).unsqueeze(0).to(DEVICE)
    src_padding_mask = (src == PAD_IDX_SRC)
    
    tgt_ids = [BOS_IDX_TGT]
    
    for _ in range(max_len):
        tgt = torch.LongTensor(tgt_ids).unsqueeze(0).to(DEVICE)
        tgt_padding_mask = (tgt == PAD_IDX_TGT)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(DEVICE)
        
        logits = model(src, tgt,
                      tgt_mask=tgt_mask,
                      src_padding_mask=src_padding_mask,
                      tgt_padding_mask=tgt_padding_mask)
        
        next_token = logits[0, -1].argmax().item()
        
        if next_token == EOS_IDX_TGT:
            break
        
        tgt_ids.append(next_token)
    
    return tokenizer_tgt.decoder.decode(
        [tokenizer_tgt.id_to_token(i) for i in tgt_ids[1:]]
    )

In [13]:
@torch.no_grad()
def translate_batch(model, texts, batch_size=16, max_len=50):
    model.eval()
    batch_size = len(texts)
    src_sequences = []
    for text in texts:
        src_ids = encode_src(text, MAX_LEN_SRC)
        src_sequences.append(torch.LongTensor(src_ids))

    src_padded = pad_sequence(src_sequences, batch_first=True, padding_value=PAD_IDX_SRC).to(DEVICE)
    src_padding_mask = (src_padded == PAD_IDX_SRC)

    tgt_sequences = torch.full((batch_size, 1), BOS_IDX_TGT, dtype=torch.long).to(DEVICE)

    finished = torch.zeros(batch_size, dtype=torch.bool).to(DEVICE)
    
    for _ in range(max_len):
        if finished.all():
            break

        tgt_padding_mask = (tgt_sequences == PAD_IDX_TGT)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_sequences.size(1)).to(DEVICE)

        logits = model(src_padded, tgt_sequences,
                      tgt_mask=tgt_mask,
                      src_padding_mask=src_padding_mask,
                      tgt_padding_mask=tgt_padding_mask)
        
        next_tokens = logits[:, -1, :].argmax(-1)

        tgt_sequences = torch.cat([tgt_sequences, next_tokens.unsqueeze(1)], dim=1)

        finished = finished | (next_tokens == EOS_IDX_TGT)

    translations = []
    for i in range(batch_size):
        tokens = tgt_sequences[i].cpu().tolist()
        if EOS_IDX_TGT in tokens:
            tokens = tokens[1:tokens.index(EOS_IDX_TGT)]
        else:
            tokens = tokens[1:]
        
        decoded = tokenizer_tgt.decoder.decode(
            [tokenizer_tgt.id_to_token(t) for t in tokens]
        )
        translations.append(decoded)
    
    return translations

In [14]:
torch.cuda.empty_cache()

In [15]:
def show_translations(model, num_examples=3):
    test_examples = [
        ("[EN] Hello, how are you?", "Привет, как дела?"),
        ("[FR] Bonjour, comment allez-vous?", "Здравствуйте, как поживаете?"),
        ("[DE] Guten Morgen", "Доброе утро"),
        ("[EN] What is your name?", "Как тебя зовут?"),
        ("[FR] Je m'appelle Pierre", "Меня зовут Пьер"),
        ("[DE] Wie geht es dir?", "Как дела?")
    ]
    
    for src_text, ref_text in test_examples[:num_examples]:
        translation = translate(src_text, model)
        print(f"Source: {src_text}")
        print(f"Translation: {translation}")
        print(f"Reference: {ref_text}\n")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX_TGT, label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

NUM_EPOCHS = 10
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1
)

best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scheduler)
    val_loss = evaluate(model, val_loader, criterion)
    
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  -> Сохранена лучшая модель (val_loss: {val_loss:.4f})")
    
    if epoch % 2 == 0:
        print("\nПримеры переводов:")
        show_translations(model)


Epoch 1/10


Train:   0%|          | 0/344 [00:00<?, ?it/s]c:\Users\Ale03x\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Eval:   0%|          | 0/47 [00:00<?, ?it/s]                         c:\Users\Ale03x\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Train Loss: 8.3557, Val Loss: 7.5526
  -> Сохранена лучшая модель (val_loss: 7.5526)

Epoch 2/10


Train Loss: 7.0580, Val Loss: 7.0568
  -> Сохранена лучшая модель (val_loss: 7.0568)

Примеры переводов:
Source: [EN] Hello, how are you?
Translation: - Да , что ты ?
Reference: Привет, как дела?

Source: [FR] Bonjour, comment allez-vous?
Translation: Ты не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не не
Reference: Здравствуйте, как поживаете?

Source: [DE] Guten Morgen
Translation: Это - то !
Reference: Доброе утро


Epoch 3/10


Train Loss: 6.5008, Val Loss: 6.8119
  -> Сохранена лучшая модель (val_loss: 6.8119)

Epoch 4/10


Train Loss: 5.9789, Val Loss: 6.7069
  -> Сохранена лучшая модель (val_loss: 6.7069)

Примеры переводов:
Source: [EN] Hello, how are you?
Translation: О , ты делаешь ?
Reference: Привет, как дела?

Source: [FR] Bonjour, comment allez-vous?
Translation: Ты же знаешь ?
Reference: Здравствуйте, как поживаете?

Source: [DE] Guten Morgen
Translation: - Амг .
Reference: Доброе утро


Epoch 5/10


... (2 outputs left out)

Train Loss: 4.1238, Val Loss: 6.9524

Epoch 8/10


Train Loss: 3.7110, Val Loss: 7.0461

Примеры переводов:
Source: [EN] Hello, how are you?
Translation: Эй , ты в порядке ?
Reference: Привет, как дела?

Source: [FR] Bonjour, comment allez-vous?
Translation: - Ты в порядке ?
Reference: Здравствуйте, как поживаете?

Source: [DE] Guten Morgen
Translation: Макакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакака
Reference: Доброе утро


Epoch 9/10


Train Loss: 3.4679, Val Loss: 7.0960

Epoch 10/10


Train Loss: 3.3682, Val Loss: 7.1026

Примеры переводов:
Source: [EN] Hello, how are you?
Translation: Эй , ты в порядке ?
Reference: Привет, как дела?

Source: [FR] Bonjour, comment allez-vous?
Translation: - Ты в порядке ?
Reference: Здравствуйте, как поживаете?

Source: [DE] Guten Morgen
Translation: Макакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакакака
Reference: Доброе утро



In [17]:
test_es_phrases = [
    "[ES] Hola, ¿cómo estás?",
    "[ES] Buenos días",
    "[ES] ¿Qué tal?",
    "[ES] Muchas gracias",
    "[ES] Hasta luego"
]

print("Переводы с испанского (не был в обучении):")
for phrase in test_es_phrases:
    translation = translate(phrase, model)
    print(f"Source: {phrase}")
    print(f"Translation: {translation}")
    print()


print("Лучшие переводы:")

for lang in ['EN', 'FR', 'DE']:
    if len(test_by_lang[lang]) >= 10:
        print(f"\nЯзык: {lang}")
        
        sample_size = min(5, len(test_by_lang[lang]))
        src_samples = test_by_lang[lang][:sample_size]
        ref_samples = ref_by_lang[lang][:sample_size]
        
        hypotheses = translate_batch(model, src_samples)
        
        scores = []
        for i, (hyp, ref) in enumerate(zip(hypotheses, ref_samples)):
            try:
                bleu = sacrebleu.sentence_bleu(hyp, [ref]).score
                scores.append((bleu, i, hyp, ref, src_samples[i]))
            except:
                continue

        scores.sort(reverse=True)
        
        if scores:
            print("Лучшие переводы:")
            for i in range(min(3, len(scores))):
                score, idx, hyp, ref, src = scores[i]
                print(f"\nBLEU: {score:.2f}")
                print(f"Source: {src}")
                print(f"Hypothesis: {hyp}")
                print(f"Reference: {ref}")

Переводы с испанского (не был в обучении):
Source: [ES] Hola, ¿cómo estás?
Translation: " О , где у тебя есть "

Source: [ES] Buenos días
Translation: [ звон ]

Source: [ES] ¿Qué tal?
Translation: " О чем это ?

Source: [ES] Muchas gracias
Translation: [ звон ]



c:\Users\Ale03x\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Source: [ES] Hasta luego
Translation: [ звон ]

Лучшие переводы:

Язык: EN
Лучшие переводы:

BLEU: 11.36
Source: [EN] How long is it going to take you to get him what he needs?
Hypothesis: Как у тебя есть что - то , что он мог бы сделать ?
Reference: Сколько времени вы будете делать то, что ему нужно?

BLEU: 10.68
Source: [EN] If you only stay there.
Hypothesis: Если ты был убит .
Reference: Только бы не вылететь.

BLEU: 4.28
Source: [EN] We might have a slight edge in mediation.
Hypothesis: Мы должны быть на пару лет .
Reference: Возможно, у нас есть небольшое преимущество в переговорах.

Язык: FR
Лучшие переводы:

BLEU: 5.00
Source: [FR] Tu sais, je l'ai senti depuis mon deuxième "chienne".
Hypothesis: Ты можешь меня сюда , чтобы я ...
Reference: Знаешь, я поняла это после второй "сучки".

BLEU: 3.31
Source: [FR] Eh bien, vous avez eu tort.
Hypothesis: - Да , ты не будешь .
Reference: Что ж, существует много других более важных событий, чем это.

BLEU: 3.12
Source: [FR] b) Une note d


## Задание 2 (2 балла).
Прочитайте главу про машинный перевод у Журафски и Маннига - https://web.stanford.edu/~jurafsky/slp3/13.pdf
Ответьте своими словами в чем заключается техника back translation? Для чего она применяется и что позволяет получить? Опишите по шагам как ее применить к паре en->ru на данных из семинара. Сколько моделей понадобится? Сколько запусков обучения нужно будет сделать?

Ответ должен содержать как минимум 10 предложений.

Техника back translation (обратный перевод) -- это способ увеличить количество данных для обучения нейросетевого переводчика, когда у нас мало примеров перевода. Смысл в том, чтобы использовать обычные тексты на одном языке и сделать из них искусственные пары предложение-перевод.

Применяют её для улучшения качества модели машинного перевода. Это метод аугментации данных, когда текст делается другим, но со схожим смыслом, чтобы модель училась на более разнообразных примерах.

Например, для пары английский-русский это работает так. Сначала мы обучаем начальную модель, которая переводит с русского на английский. Затем берём большой массив русских текстов и прогоняем их через эту модель RU->EN. На выходе получаем корявенькие английские переводы для каждого русского предложения. Склеиваем эти новые данные и получается искусственная параллель плохой английский / оригинальный русский. Добавляем эту синтетику к исходным правильным примерам. На расширенном датасете тренируем финальную нужную модель с английского на русский. Нам понадобятся две модели: одна для обратного перевода (ru->en) и одна для целевого (en->ru). Что касается запусков, то сначала мы один раз обучаем "обратную" модель, затем мы один раз прогоняем через неё все тексты (инференс — это не обучaение), а потом финально один раз обучаем целевую модель. Иногда этот процесс могут повторять по кругу несколько раз, улучшая результаты с каждой итерацией, но в базовом варианте достаточно одного цикла. Получается, что мы обманули систему и сделали из односторонних текстов полноценный двуязычный корпус.